OpenGAN: Open-Set Recognition via Open Data Generation
================
**Supplemental Material for ICCV2021 Submission**


In this notebook, we demonstrate how we train the GAN-fea model on the TinyImageNet train-set, providing the closed-set images.

import packages
------------------

Some packages are installed automatically through Anaconda. PyTorch should be also installed.

In [11]:
from __future__ import print_function, division
import os, random, time, copy
#from skimage import io, transform

import os
import sys

import matplotlib.pyplot as plt

#from skimage import data, img_as_float
#from skimage.measure import compare_ssim as ssim
#from skimage.measure import compare_psnr as psnr

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable

from torchvision import datasets, models, transforms
import torchvision.utils as vutils

# from utils.eval_funcs import *
# from utils.dataset_tinyimagenet import *

import warnings # ignore warnings
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)

sys.path.append(os.path.abspath(".."))
from Utils import *
from Utils_OpenGan import *
manualSeed = 42
fix_random_seed(manualSeed)

3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
2.9.1+cu128


Setup config parameters
 -----------------
 
 There are several things to setup, like which GPU to use, where to read images and save files, etc. Please read and understand this. By default, you should be able to run this script smoothly by changing nothing.

In [12]:

################## set attributes for this project/experiment ##################
# config result folder
exp_dir = './Experimentos/Mnist_omni/' # experiment directory, used for reading the init model

modelFlag = "ResNet18"

project_name = 'OpenGan_Mnist_omni' + modelFlag   # we save all the checkpoints in this directory

# set device, which gpu to use.
device ='cpu'
if torch.cuda.is_available(): 
    device='cuda:0'


total_epoch_num = 200 # total number of epoch in training
batch_size = 256    

newsize = (64, 64)


path_to_feats = os.path.join(NOMES.FEATS_DIR.value, NOMES.MNIST_OMNI.value,"ResNet18") # the path to cached off-the-shelf features

# For GAN-fea, we set the hyper-parameters as below.

# Number of channels in the training images. For color images this is 3
nc = 512

# Size of z latent vector (i.e. size of generator input)
nz = 80

# Size of feature maps in generator
ngf = 128

# Size of feature maps in discriminator
ndf = 128

# Beta1 hyperparam for Adam optimizers
beta1 = 0.5

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1



# TinyImageNet has 200 classes.
nClassTotal = 10
nClassCloseset = nClassTotal

#project_name += '_K{}run{}'.format(nClassCloseset, runIdx)
if not os.path.exists(exp_dir): os.makedirs(exp_dir)

lr = 0.0001 # learning rate

num_epochs = total_epoch_num
torch.cuda.device_count()
torch.cuda.empty_cache()

save_dir = os.path.join(exp_dir, project_name)
print(save_dir)    
print(path_to_feats)
if not os.path.exists(save_dir): os.makedirs(save_dir)

log_filename = os.path.join(save_dir, 'train.log')

./Experimentos/Mnist_omni/OpenGan_Mnist_omniResNet18
/home/alexandreselani/Desktop/Features_extraidas/Mnist_Omni/ResNet18


In [13]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)     
        

class Generator(nn.Module):
    def __init__(self, ngpu=1, nz=100, ngf=64, nc=512):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.nc = nc
        
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            # Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros')
            nn.Conv2d( self.nz, self.ngf * 8, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ngf * 8),
            nn.ReLU(True),
            # state size. (self.ngf*8) x 4 x 4
            nn.Conv2d(self.ngf * 8, self.ngf * 4, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ngf * 4),
            nn.ReLU(True),
            # state size. (self.ngf*4) x 8 x 8
            nn.Conv2d( self.ngf * 4, self.ngf * 2, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ngf * 2),
            nn.ReLU(True),
            # state size. (self.ngf*2) x 16 x 16
            nn.Conv2d( self.ngf * 2, self.ngf*4, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ngf*4),
            nn.ReLU(True),
            # state size. (self.ngf) x 32 x 32
            nn.Conv2d( self.ngf*4, self.nc, 1, 1, 0, bias=True),
            #nn.Tanh()
            # state size. (self.nc) x 64 x 64
        )

    def forward(self, input):
        return self.main(input)

    
class Discriminator(nn.Module):
    def __init__(self, ngpu=1, nc=512, ndf=64):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.nc = nc
        self.ndf = ndf
        self.main = nn.Sequential(
            nn.Conv2d(self.nc, self.ndf*8, 1, 1, 0, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(self.ndf*8, self.ndf*4, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(self.ndf*4, self.ndf*2, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(self.ndf*2, self.ndf, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ndf),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(self.ndf, 1, 1, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

## initialize the networks

In [14]:
netG = Generator(ngpu=ngpu, nz=nz, ngf=ngf, nc=nc).to(device)
netD = Discriminator(ngpu=ngpu, nc=nc, ndf=ndf).to(device)

# Handle multi-gpu if desired
if ('cuda' in device) and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))

    # Apply the weights_init function to randomly initialize all weights
#  to mean=0, stdev=0.2.
netD.apply(weights_init)


if ('cuda' in device) and (ngpu > 1):
    netG = nn.DataParallel(netG, list(range(ngpu)))
netG.apply(weights_init)

print(device)

cuda:0


## Sanity Check

In [15]:
noise = torch.randn(batch_size, nz, 1, 1, device=device)
# Generate fake image batch with G
fake = netG(noise)
predLabel = netD(fake)

print(noise.shape, fake.shape, predLabel.shape)

torch.Size([256, 80, 1, 1]) torch.Size([256, 512, 1, 1]) torch.Size([256, 1, 1, 1])


## setup dataset of TinyImageNet

In [16]:
mnist_train_data = torch.load(os.path.join(path_to_feats,"mnist_treino_features.pt"),map_location='cpu')



## a helper function to read cached off-the-shelf features per closed images

In [17]:
class FeatDataset(Dataset):
    def __init__(self, data):
        self.features = data["features"]
        self.labels = data["labels"]
        self.current_set_len = self.features.shape[0]        
        
    def __len__(self):        
        return self.current_set_len
    
    def __getitem__(self, idx):        
        return self.features[idx]
    
    def get_label(self,idx):
        return self.labels[idx]

In [18]:
trainset_closeset = FeatDataset(data=mnist_train_data)
print(trainset_closeset.__getitem__(2))
dataloader = DataLoader(trainset_closeset, batch_size=batch_size, shuffle=True, num_workers=1)

tensor([0.0000e+00, 5.5141e-04, 2.7845e-02, 2.1744e-03, 2.3725e-03, 1.1208e+00,
        1.8987e-03, 1.5415e-03, 1.4580e-01, 1.1521e+00, 0.0000e+00, 9.2784e-02,
        7.5127e-01, 0.0000e+00, 6.4984e-03, 1.4018e-02, 1.9727e-02, 2.4228e-03,
        9.1266e-01, 4.6221e-01, 0.0000e+00, 3.4971e-03, 1.4403e-01, 4.8010e-02,
        2.4877e-01, 5.9363e-01, 5.7488e-03, 1.7751e-02, 6.8855e-01, 0.0000e+00,
        9.1748e-01, 5.3803e-01, 1.9785e-02, 1.2336e-02, 2.5388e-02, 2.8064e-02,
        3.3863e-03, 3.4912e-01, 1.4068e-02, 3.9206e-03, 0.0000e+00, 0.0000e+00,
        2.2961e-03, 4.1690e-01, 7.7808e-04, 3.5867e-04, 1.0794e+00, 7.8623e-04,
        6.0866e-03, 2.8453e-03, 0.0000e+00, 3.2247e-02, 0.0000e+00, 2.8457e-03,
        0.0000e+00, 5.2908e-01, 7.1177e-03, 0.0000e+00, 3.3348e-03, 1.7834e-03,
        8.0798e-03, 6.7243e-01, 7.7281e-03, 1.0399e-02, 7.3412e-04, 3.1429e-03,
        0.0000e+00, 7.2574e-03, 1.1646e-01, 3.4369e-03, 2.0642e-01, 1.3099e-03,
        2.4276e-03, 1.0771e-01, 1.4664e-

setup training
-----------

In [19]:
# Initialize BCELoss function
criterion = nn.BCELoss()

# Create batch of latent vectors that we will use to visualize
#  the progression of the generator
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Establish convention for real and fake labels during training
real_label = 1
fake_label = 0

# Setup Adam optimizers for both G and D
optimizerD = optim.Adam(netD.parameters(), lr=lr/1.5, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

In [20]:
# Training Loop

from utils.eval_funcs import *
import gc

# ---- dados de validacao, usados para avaliar o discriminador a cada epoca ----
mnist_val_closedset = torch.load(os.path.join(path_to_feats,"mnist_val_features.pt"))
mnist_val_closedset_dataset = FeatDataset(mnist_val_closedset)
dataloader_val_closedset = DataLoader(mnist_val_closedset_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

omniglot_val_openset = torch.load(os.path.join(path_to_feats,"omni_val_features.pt"))
omniglot_val_openset_dataset = FeatDataset(omniglot_val_openset)
dataloader_val_openset = DataLoader(omniglot_val_openset_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


def confidencias(netD, loader):
    """Confianca do discriminador para todas as amostras do loader."""
    confs = torch.tensor([]).type(torch.float)
    for X in loader:
        X = X.to(device, dtype=torch.float32)
        feats = X.view(-1, X.shape[-1], 1, 1)
        with torch.no_grad():
            predConf = netD(feats)
        confs = torch.cat((confs, predConf.reshape(-1).detach().cpu()), 0)
    return confs.numpy()


def avalia_auroc(netD):
    """AUROC do discriminador separando conhecidas (mnist) de desconhecidas (omniglot)."""
    netD.eval()
    conf_close = confidencias(netD, dataloader_val_closedset)
    conf_open = confidencias(netD, dataloader_val_openset)
    netD.train()
    roc_score, _ = evaluate_openset(-conf_close, -conf_open)
    return roc_score


# Melhor discriminador visto ate agora, pela AUROC de validacao.
melhor_auroc = -1
melhor_epoch = -1
path_best_D = os.path.join(save_dir, 'best.DNet')

# Lists to keep track of progress
img_list = []
G_losses = []
D_losses = []
aurocs = []
iters = 0

print("Starting Training Loop...")
# For each epoch
for epoch in range(num_epochs):
    # For each batch in the dataloader
    for i, data in enumerate(dataloader, 0):
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        ## Train with all-real batch
        netD.zero_grad()
        # Format batch
        real_cpu = data.to(device)
        real_cpu = real_cpu.view(real_cpu.size(0), nc, 1, 1).to(device,dtype=torch.float32)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, device=device,dtype=torch.float32)
        # Forward pass real batch through D
        output = netD(real_cpu).view(-1)
        #print(output.shape)
        # Calculate loss on all-real batch
        errD_real = criterion(output, label)
        # Calculate gradients for D in backward pass
        errD_real.backward()
        D_x = output.mean().item()

        ## Train with all-fake batch
        # Generate batch of latent vectors
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        # Generate fake image batch with G
        fake = netG(noise)
        label.fill_(fake_label)
        # Classify all fake batch with D
        output = netD(fake.detach()).view(-1)
        # Calculate D's loss on the all-fake batch
        errD_fake = criterion(output, label)
        # Calculate the gradients for this batch
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        # Add the gradients from the all-real and all-fake batches
        errD = errD_real + errD_fake
        # Update D
        optimizerD.step()
        

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netG.zero_grad()
        label.fill_(real_label)  # fake labels are real for generator cost
        # Since we just updated D, perform another forward pass of all-fake batch through D
        output = netD(fake).view(-1)
        # Calculate G's loss based on this output
        errG = criterion(output, label)
        # Calculate gradients for G
        errG.backward()
        D_G_z2 = output.mean().item()
        # Update G
        optimizerG.step()

        # Output training stats
        if i % 200 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        # Save Losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        # Check how the generator is doing by saving G's output on fixed_noise
        if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

        iters += 1
        
    
    # ---- avaliacao ao fim da epoca: grava so se a AUROC melhorou ----
    gc.collect()
    torch.cuda.empty_cache()

    auroc = avalia_auroc(netD)
    aurocs.append(auroc)

    if auroc > melhor_auroc:
        melhor_auroc = auroc
        melhor_epoch = epoch + 1
        torch.save(copy.deepcopy(netD.state_dict()), path_best_D)
        print('epoch {}: AUROC {:.5f}  <-- novo melhor, gravado'.format(epoch+1, auroc))
    else:
        print('epoch {}: AUROC {:.5f}  (melhor: {:.5f} na epoca {})'.format(
            epoch+1, auroc, melhor_auroc, melhor_epoch))

print('\nMelhor discriminador: epoca {} (AUROC {:.5f})'.format(melhor_epoch, melhor_auroc))
print('Gravado em: {}'.format(path_best_D))

Starting Training Loop...
[0/200][0/211]	Loss_D: 1.3900	Loss_G: 0.6962	D(x): 0.5012	D(G(z)): 0.5011 / 0.4996


[0/200][200/211]	Loss_D: 1.2089	Loss_G: 0.7393	D(x): 0.5759	D(G(z)): 0.4809 / 0.4780
epoch 1: AUROC 0.85654  <-- novo melhor, gravado
[1/200][0/211]	Loss_D: 1.2053	Loss_G: 0.7405	D(x): 0.5771	D(G(z)): 0.4803 / 0.4774
[1/200][200/211]	Loss_D: 1.1238	Loss_G: 0.7949	D(x): 0.5988	D(G(z)): 0.4564 / 0.4523
epoch 2: AUROC 0.72351  (melhor: 0.85654 na epoca 1)
[2/200][0/211]	Loss_D: 1.1183	Loss_G: 0.7995	D(x): 0.5996	D(G(z)): 0.4543 / 0.4502
[2/200][200/211]	Loss_D: 1.0300	Loss_G: 0.8636	D(x): 0.6256	D(G(z)): 0.4282 / 0.4226
epoch 3: AUROC 0.64230  (melhor: 0.85654 na epoca 1)
[3/200][0/211]	Loss_D: 1.0386	Loss_G: 0.8674	D(x): 0.6184	D(G(z)): 0.4268 / 0.4209
[3/200][200/211]	Loss_D: 0.9463	Loss_G: 0.9463	D(x): 0.6462	D(G(z)): 0.3975 / 0.3897
epoch 4: AUROC 0.54417  (melhor: 0.85654 na epoca 1)
[4/200][0/211]	Loss_D: 0.9304	Loss_G: 0.9477	D(x): 0.6551	D(G(z)): 0.3965 / 0.3890
[4/200][200/211]	Loss_D: 0.8375	Loss_G: 1.0538	D(x): 0.6789	D(G(z)): 0.3607 / 0.3506
epoch 5: AUROC 0.62578  (melhor: 0.

## drawing the error curves

In [21]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses,label="G")
plt.plot(D_losses,label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.savefig('learningCurves_{}.png'.format(modelFlag), bbox_inches='tight',transparent=True)
plt.show()

# Validation

In [22]:
import utils
print(utils.__file__)

/home/alexandreselani/Desktop/OpenGan/OpenGAN-IC/utils/__init__.py


In [23]:
# from utils.eval_funcs import *
# import gc
# maiores_rocs = []
# melhores_epochs = []


# mnist_val_closedset  = torch.load(os.path.join(path_to_feats,"mnist_val_features.pt"))
# mnist_val_closedset_dataset = FeatDataset(mnist_val_closedset)
# dataloader_val_closedset = DataLoader(mnist_val_closedset_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# omniglot_val_openset = torch.load(os.path.join(path_to_feats,"omni_val_features.pt"))
# omniglot_val_openset_dataset = FeatDataset(omniglot_val_openset)
# dataloader_val_openset = DataLoader(omniglot_val_openset_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# maior_roc_iteracao = -1
# melhor_epoch = -1

# for epoch in range(num_epochs):
#     print(f"EPOCH {epoch}")
#     gc.collect()
#     torch.cuda.empty_cache()

    
#     netD = Discriminator(ngpu=ngpu, nc=nc, ndf=ndf).to(device)
#     path_to_D = os.path.join(save_dir,'epoch-{}.DNet'.format(epoch+1))
    
#     netD.load_state_dict(torch.load(path_to_D))
#     netD.eval()
    

#     feat_close_mnist = torch.tensor([]).type(torch.float)
#     label_close_mnist = torch.tensor([]).type(torch.float)
#     conf_close_mnist = torch.tensor([]).type(torch.float)

#     i = 0
#     count = 0

#     for X in dataloader_val_closedset:  

#         X = X[0].to(device,dtype=torch.float32)
#         # y = y.type(torch.long).view(-1).to(device)    
#         count += X.shape[0]
#         i+=1 
#         feats = X.to(device, dtype=torch.float32).view(-1, X.shape[-1], 1, 1)
#         #print(feats)
#         with torch.no_grad():
#             predConf = netD(feats)
#         predConf = predConf.view(-1,1)
#         #print(predConf[0,0])
#         conf_close_mnist = torch.cat((conf_close_mnist, predConf.reshape(-1).detach().cpu()), 0)
        
#         feats = feats.squeeze()
#         #label_close_mnist = torch.cat((label_close_mnist, y.type(torch.float).detach().cpu().reshape(-1,1))) 
            
#     conf_close_mnist = conf_close_mnist.detach().cpu().numpy()
# # We draw the ROC curve for classifying closed-set and open-set data.
#     feat_open_omniglot = torch.tensor([]).type(torch.float)
#     label_open_omniglot = torch.tensor([]).type(torch.float)
#     conf_open_omniglot = torch.tensor([]).type(torch.float)

#     i = 0
#     count = 0
#     for X in dataloader_val_openset:
#         X = X[0].to(device,dtype=torch.float32)    
#         count += X.shape[0]
#         i+=1    
#         feats = X.to(device, dtype=torch.float32).view(-1, X.shape[-1], 1, 1)
        
#         with torch.no_grad():
#             predConf = netD(feats)        
#         predConf = predConf.view(-1,1).detach()
#         conf_open_omniglot = torch.cat((conf_open_omniglot, predConf.reshape(-1).detach().cpu()), 0)
#         #print(conf_open_cifar10)
#         feats = feats.squeeze()
#         feat_open_omniglot = torch.cat((feat_open_omniglot, feats.detach().cpu()))
#         #label_open_cifar10 = torch.cat((label_open_cifar10, torch.Tensor(-1))) 
        

#     conf_open_omniglot = conf_open_omniglot.detach().cpu().numpy()    

#     roc_score, roc_to_plot = evaluate_openset(-conf_close_mnist, -conf_open_omniglot)

#     plt.plot(roc_to_plot['fp'], roc_to_plot['tp'])
#     plt.grid('on')
#     plt.xlabel('False Positive Rate')
#     plt.ylabel('True Positive Rate')
#     plt.title('ROC score {:.5f}'.format(roc_score))
            
#     print(roc_score)
#     if(roc_score>maior_roc_iteracao):
#         maior_roc_iteracao=roc_score
#         melhor_epoch = epoch+1

# maiores_rocs.append(maior_roc_iteracao)
# melhores_epochs.append(melhor_epoch)

# for iter, roc in enumerate(maiores_rocs):
#     print(f"Maior roc na Iteração {iter}: {roc} (epoch {melhores_epochs[iter]})")


# print(f"roc media = {np.array(maiores_rocs).mean()}")